# Movie split, user EDA and train-only aggregates

**Goal.** Split the data for the rating model and look at the users before clustering them.

- **Split by movie** (70% train / 15% validation / 15% test). The project targets *cold start*: predicting ratings for a movie nobody has rated yet, from its TMDB metadata. Validation and test movies are therefore never seen in training; all users can appear in all three parts.
- **User EDA** on training-movie ratings: how many ratings users have, how much of a rating is explained by the user vs. the movie, whether users differ in genre taste (the premise of persona clustering), and the rating distribution behind the later "add to list" threshold.
- **Train-only aggregates** for the next notebooks: movie rating statistics, director / cast track records and user statistics, all computed from **training movies only** so nothing leaks from validation / test movies.

## 1. Load

Movie features come from `06_movie_features.ipynb`. Ratings (~57M rows) are read in chunks with compact integer types to keep memory low; the `date` column is not needed.

In [ ]:
import ast
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PROCESSED = Path("../data/processed")
RANDOM_STATE = 42

movies = pd.read_csv(PROCESSED / "movies_features.csv", converters={"cast_top5": ast.literal_eval})
GENRE_COLS = [c for c in movies.columns if c.startswith("genre_")]

chunks = pd.read_csv(
    PROCESSED / "ratings.csv",
    usecols=["netflix_movie_id", "rater_id", "rating"],
    dtype={"netflix_movie_id": "int32", "rater_id": "int32", "rating": "int8"},
    chunksize=5_000_000,
)
ratings = pd.concat(chunks, ignore_index=True)
n_all = len(ratings)
ratings = ratings[ratings["netflix_movie_id"].isin(movies["netflix_movie_id"])].reset_index(drop=True)

print(f"movies: {movies.shape}")
print(f"ratings: {len(ratings):,} (dropped {n_all - len(ratings):,} of movies without features)")
print(f"raters: {ratings['rater_id'].nunique():,}")

## 2. Movie split

Movies are split 70 / 15 / 15, **stratified** so that the three parts look alike:
- popularity (tercile of the number of ratings) × era (before 1980, 1980s, 1990s, 2000s): popular recent blockbusters and rarely rated old films should appear in every part;
- documentaries form their own stratum (only 10 movies), so some of them are in training and documentary taste can enter the persona profiles.

The number of ratings is used only to stratify, not as a feature.

In [ ]:
movie_n = ratings.groupby("netflix_movie_id").size().rename("n_ratings")
movie_mean = ratings.groupby("netflix_movie_id")["rating"].mean().rename("avg_rating")
movies = movies.join(movie_n, on="netflix_movie_id").join(movie_mean, on="netflix_movie_id")

era = pd.cut(movies["year"], [0, 1979, 1989, 1999, 2005], labels=["<1980", "1980s", "1990s", "2000s"])
popularity = pd.qcut(movies["n_ratings"], 3, labels=["low", "mid", "high"])
strata = np.where(movies["genre_documentary"] == 1, "documentary", popularity.astype(str) + "_" + era.astype(str))
print("smallest strata:", pd.Series(strata).value_counts().tail(4).to_dict())

ids = movies["netflix_movie_id"].to_numpy()
train_ids, rest_ids, _, rest_strata = train_test_split(
    ids, strata, test_size=0.30, stratify=strata, random_state=RANDOM_STATE
)
val_ids, test_ids = train_test_split(
    rest_ids, test_size=0.50, stratify=rest_strata, random_state=RANDOM_STATE
)

split_of = {**{m: "train" for m in train_ids}, **{m: "val" for m in val_ids}, **{m: "test" for m in test_ids}}
movies["split"] = movies["netflix_movie_id"].map(split_of)
assert movies["split"].notna().all()
assert not (set(train_ids) & set(val_ids) or set(train_ids) & set(test_ids) or set(val_ids) & set(test_ids))
movies["split"].value_counts()

**Balance check.** The three parts should have similar ratings, popularity, age and genre mix. (`avg_rating` of validation / test movies is used here only as a check, never as a feature.)

In [ ]:
balance = movies.groupby("split").agg(
    movies=("netflix_movie_id", "size"),
    ratings=("n_ratings", "sum"),
    median_n_ratings=("n_ratings", "median"),
    avg_rating=("avg_rating", "mean"),
    year=("year", "mean"),
    runtime=("runtime", "mean"),
    log_roi=("log_roi", "mean"),
    drama=("genre_drama", "mean"),
    comedy=("genre_comedy", "mean"),
    horror=("genre_horror", "mean"),
    documentaries=("genre_documentary", "sum"),
    us_only=("us_only", "mean"),
).loc[["train", "val", "test"]]
balance["pct_ratings"] = 100 * balance["ratings"] / balance["ratings"].sum()
balance.round(2)

Every rating gets the split of its movie. A lookup array indexed by movie id is used instead of `map` so the 57M-row column stays one byte per row.

In [ ]:
SPLITS = np.array(["train", "val", "test"])
split_lut = np.full(ratings["netflix_movie_id"].max() + 1, -1, dtype="int8")
for code_, name in enumerate(SPLITS):
    split_lut[movies.loc[movies["split"] == name, "netflix_movie_id"].to_numpy()] = code_
ratings["split"] = split_lut[ratings["netflix_movie_id"].to_numpy()]
assert (ratings["split"] >= 0).all()

train_ratings = ratings[ratings["split"] == 0]
print({name: f"{int((ratings['split'] == i).sum()):,}" for i, name in enumerate(SPLITS)})

movies[["netflix_movie_id", "split"]].to_csv(PROCESSED / "movie_split.csv", index=False)

## 3. Ratings per user on training movies

A user's profile and persona are built from their ratings of **training movies** only, so a minimum number of such ratings is needed for a stable profile. The table shows, for each threshold, how many users qualify and what share of the validation / test ratings belongs to them (the ratings the model can be evaluated on).

In [ ]:
train_n = train_ratings.groupby("rater_id").size()
eval_n = ratings[ratings["split"] > 0].groupby("rater_id").size()
n_users = ratings["rater_id"].nunique()

rows = []
for min_n in [5, 10, 20, 30, 50, 100]:
    kept = train_n.index[train_n >= min_n]
    rows.append({
        "min_train_ratings": min_n,
        "users": len(kept),
        "pct_users": 100 * len(kept) / n_users,
        "pct_val_test_ratings": 100 * eval_n.reindex(kept).sum() / eval_n.sum(),
    })
display(pd.DataFrame(rows).set_index("min_train_ratings").round(1))

bins = [0, 4, 9, 19, 49, 99, 249, np.inf]
labels = ["1-4", "5-9", "10-19", "20-49", "50-99", "100-249", "250+"]
counts = pd.cut(train_n, bins=bins, labels=labels).value_counts().reindex(labels)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(labels, counts.values, color="steelblue")
ax.set_xlabel("ratings of training movies per user")
ax.set_ylabel("users")
ax.set_title(f"Training ratings per user (median {train_n.median():.0f})")
fig.tight_layout()
plt.show()

`MIN_TRAIN_RATINGS` is the threshold used from here on (user statistics, clustering, model). 20 ratings give a usable per-genre profile while keeping most users and most evaluation ratings; adjust it if the table above suggests otherwise.

In [ ]:
MIN_TRAIN_RATINGS = 20

eligible_users = train_n.index[train_n >= MIN_TRAIN_RATINGS]
print(f"eligible users: {len(eligible_users):,} of {n_users:,}")

## 4. User vs. movie: how much of a rating does each explain?

If users differ strongly in how generous they are, the user's mean rating is a key model feature. To measure this, each rating is "predicted" by a simple rule, and the rule is scored with **R²**:

$$R^2 = 1 - rac{\sum (	ext{rating} - 	ext{prediction})^2}{\sum (	ext{rating} - 	ext{average rating})^2}$$

- The denominator is the total variation of the ratings: the squared error of the dumbest prediction, "every rating equals the overall average".
- The numerator is the squared error left after using the rule.
- So R² is the **share of the rating variation the rule explains**: 0 = no better than the overall average, 1 = every rating predicted exactly. For example, R² = 0.16 means the rule removes 16% of the squared error.

The three rules:
- **movie mean**: every rating is predicted by the movie's average rating (how good the movie is),
- **user mean**: every rating is predicted by the user's average rating (how generous the user is),
- **user + movie bias**: the movie's average plus the user's average offset from the movies they rated (how good the movie is, adjusted for how generous the user is; the classic recommender baseline).

Computed on training ratings of a random sample of 50,000 eligible users (to keep memory low); every rating of a sampled user is used.

In [ ]:
N_SAMPLE_USERS = 50_000

rng = np.random.default_rng(RANDOM_STATE)
sample_users = rng.choice(eligible_users.to_numpy(), size=min(N_SAMPLE_USERS, len(eligible_users)), replace=False)
sample = train_ratings[train_ratings["rater_id"].isin(sample_users)].copy()
sample["rating"] = sample["rating"].astype(float)

global_mean = train_ratings["rating"].mean()
train_movie_mean = train_ratings.groupby("netflix_movie_id")["rating"].mean()

sample["movie_mean"] = sample["netflix_movie_id"].map(train_movie_mean)
sample["user_mean"] = sample.groupby("rater_id")["rating"].transform("mean")
user_offset = (sample["rating"] - sample["movie_mean"]).groupby(sample["rater_id"]).transform("mean")
sample["user_movie_bias"] = sample["movie_mean"] + user_offset

total_var = ((sample["rating"] - sample["rating"].mean()) ** 2).mean()
r2 = {
    name: 1 - ((sample["rating"] - sample[col]) ** 2).mean() / total_var
    for name, col in [("movie mean", "movie_mean"), ("user mean", "user_mean"), ("user + movie bias", "user_movie_bias")]
}
print(f"sample: {len(sample_users):,} users, {len(sample):,} ratings")
pd.Series(r2, name="R²").round(3)

## 5. Do users differ in genre taste?

Persona clustering only makes sense if users differ in *which* genres they like, beyond being generous or harsh overall.

**Genre score of a user.** For each user and genre (only user–genre pairs with ≥ 5 ratings):

$$	ext{genre score} = 	ext{the user's average rating of that genre's films} - 	ext{the user's average rating of all films}$$

It is a **difference between two ratings, not a rating**, so it is not on the 1–5 scale: it is centred at 0 and in practice almost always lies between −1.5 and +1.5 stars.
- Example: a user whose overall average is 3.5 and who gives action films 4.0 on average has an action score of 4.0 − 3.5 = **+0.5** (likes action more than their usual film); with 3.0 on action films the score is **−0.5**; 0 means action is rated like everything else.
- Subtracting the user's own average removes generosity: a harsh user (average 2.5, action 3.0) and a generous user (average 4.0, action 4.5) both get +0.5, because both like action equally relative to how they usually rate.

**Why a comparison is needed.** Even if all users had exactly the same taste, their genre scores would still differ a bit, just by chance (which films of the genre they happened to rate, and rating noise). So a spread of genre scores alone proves nothing. To see how much spread chance alone produces, the scores are recomputed after **shuffling each user's ratings across their own movies**: each user keeps their ratings and their movies, but which rating goes with which movie is random, so any real taste is destroyed.

**Spread.** The spread of a set of genre scores is their **standard deviation**: how far a typical user's score lies from the average score, i.e. how wide the histogram is. The *spread ratio* compares the real users with the shuffled ones:

$$	ext{spread ratio} = rac{	ext{spread of the real genre scores}}{	ext{spread of the shuffled genre scores (luck only)}}$$

Example, action: real users' action scores have a spread of 0.29 stars, luck alone gives 0.21, so the ratio is 0.29 / 0.21 = **1.38**: users disagree about action 38% more than luck would produce. This is the "*X*% more spread than luck" in the histogram titles (38% = 1.38 − 1). A ratio of **1** means users do not disagree about the genre beyond luck.

- **Plot 1 (histograms, 6 example genres):** x-axis = the genre score, y-axis = % of users with that score; real users in blue, shuffled users (luck only) as a grey outline. Top row: the 3 genres with the highest spread ratio; bottom row: the 3 lowest. If the blue histogram is **wider** than the grey one, users really disagree about that genre: some love it, some dislike it. If the two overlap, the genre does not divide users. A blue histogram **shifted** to the right of the grey one (documentary, war) is a different signal: almost every user rates that genre above their own average, so it is a genre everyone likes, not one that divides users. The shuffled histogram is always centred at 0 because shuffling removes any genre effect.
- **Plot 2 (bar chart, all 18 genres):** the spread ratio of every genre, sorted. Long bars are the genres that split users into fans and non-fans and will separate the personas; bars near **1** (red line) are genres users do not really disagree about.

In [ ]:
MIN_GENRE_RATINGS = 5

sample["dev"] = sample["rating"] - sample["user_mean"]
# permute deviations within each user: sort by user, then shuffle inside each user's block
sample = sample.sort_values("rater_id")
block_start = np.r_[0, np.flatnonzero(np.diff(sample["rater_id"].to_numpy())) + 1, len(sample)]
dev = sample["dev"].to_numpy()
dev_shuffled = np.concatenate([rng.permutation(dev[a:b]) for a, b in zip(block_start[:-1], block_start[1:])])
sample["dev_shuffled"] = dev_shuffled

genre_matrix = movies.set_index("netflix_movie_id")[GENRE_COLS]
g = genre_matrix.reindex(sample["netflix_movie_id"]).to_numpy()
rows, genre_scores = [], {}
for j, genre in enumerate(GENRE_COLS):
    in_genre = g[:, j] == 1
    part = sample.loc[in_genre, ["rater_id", "dev", "dev_shuffled"]]
    per_user = part.groupby("rater_id").agg(n=("dev", "size"), real=("dev", "mean"), shuffled=("dev_shuffled", "mean"))
    per_user = per_user[per_user["n"] >= MIN_GENRE_RATINGS]
    genre_scores[genre.removeprefix("genre_")] = per_user
    rows.append({
        "genre": genre.removeprefix("genre_"),
        "users": len(per_user),
        "mean_dev": per_user["real"].mean(),
        "spread_real": per_user["real"].std(),
        "spread_shuffled": per_user["shuffled"].std(),
    })
taste = pd.DataFrame(rows).set_index("genre")
taste["ratio"] = taste["spread_real"] / taste["spread_shuffled"]
taste = taste.sort_values("ratio", ascending=False)

# the 3 genres that divide users most and the 3 that divide them least
examples = list(taste.index[:3]) + list(taste.index[-3:])
bins = np.linspace(-1.5, 1.5, 41)
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True)
for ax, genre in zip(axes.ravel(), examples):
    scores = genre_scores[genre]
    pct = np.full(len(scores), 100 / len(scores))  # each user counts as a share of all users -> y-axis in %
    ax.hist(scores["real"], bins=bins, weights=pct, color="steelblue", alpha=0.7, label="real users")
    ax.hist(scores["shuffled"], bins=bins, weights=pct, histtype="step", color="gray", lw=1.5, label="shuffled (luck only)")
    ax.axvline(0, color="black", lw=0.6)
    wider = 100 * (taste.loc[genre, "ratio"] - 1)
    ax.set_title(f"{genre}: real users {wider:.0f}% more spread than luck", fontsize=10)
for ax in axes[:, 0]:
    ax.set_ylabel("% of users")
for ax in axes[1]:
    ax.set_xlabel("genre score = genre average − user's overall average (stars)")
axes[0, 0].legend(fontsize=8)
fig.suptitle("How users score a genre relative to their own average — top: genres that divide users most, bottom: least")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(taste.index[::-1], taste["ratio"][::-1], color="steelblue")
for y, v in enumerate(taste["ratio"][::-1]):
    ax.annotate(f"{v:.2f}", (v, y), xytext=(3, -3), textcoords="offset points", fontsize=7)
ax.axvline(1, color="#c44e52", ls="--", lw=1, label="1 = users differ no more than luck")
ax.set_xlim(0.9, taste["ratio"].max() + 0.08)
ax.set_xlabel("spread of real genre scores ÷ spread under luck only")
ax.set_title("How much more users disagree about each genre than luck would produce")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()
taste.round(3)

## 6. Popularity and age vs. rating

Per training movie: is a movie's average rating related to how often it is rated (popularity) and to its age? Popularity is not a model feature (it is unknown for a new movie), but it shows whether frequently watched films are rated differently.

In [ ]:
train_movies = movies[movies["split"] == "train"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label in [
    (axes[0], "n_ratings", "number of ratings (log scale)"),
    (axes[1], "movie_age_log", "movie_age_log"),
]:
    ax.scatter(train_movies[col], train_movies["avg_rating"], s=6, alpha=0.4, color="steelblue")
    rho = train_movies[col].corr(train_movies["avg_rating"], method="spearman")
    ax.set_title(f"avg_rating vs {col} (Spearman {rho:.2f})", fontsize=10)
    ax.set_xlabel(label)
    ax.set_ylabel("avg_rating")
axes[0].set_xscale("log")
fig.tight_layout()
plt.show()

pd.concat({
    "by popularity quartile": train_movies.groupby(pd.qcut(train_movies["n_ratings"], 4, labels=["Q1 (least rated)", "Q2", "Q3", "Q4 (most rated)"]), observed=True)["avg_rating"].agg(["size", "mean"]),
    "by era": train_movies.groupby(pd.cut(train_movies["year"], [0, 1979, 1989, 1999, 2005], labels=["<1980", "1980s", "1990s", "2000s"]), observed=True)["avg_rating"].agg(["size", "mean"]),
}).round(2)

**Observations**

- **Popular movies are rated higher** (Spearman 0.43): the average rating rises steadily from 3.21 for the least rated quarter of movies to 3.66 for the most rated quarter. Part of this is quality attracting viewers, part is selection: people mostly watch films they expect to like. Popularity is not a model feature (unknown for a new movie), but it means the model will see many more ratings of good, popular films than of weak ones.
- **Older movies are rated higher** (Spearman 0.27): 3.75 for pre-1980 films vs. 3.35 for 2000s films. Old films that are still on offer are the classics that survived; recent films include the weak ones. This is a survivorship effect rather than "old is better", but it makes `movie_age_log` a useful feature.

## 7. Rating distribution — background for the "add to list" threshold

The model predicts a 1–5 rating and a movie is added to the list when the prediction passes a threshold chosen later. The share of ratings at each star level, and how much it varies between users, shows what a threshold such as 4 means in practice.

In [ ]:
dist = ratings["rating"].value_counts(normalize=True).sort_index()
print(f"share of ratings >= 4: {dist.loc[[4, 5]].sum():.1%} | = 5: {dist.loc[5]:.1%}")

user_ge4 = (train_ratings["rating"] >= 4).groupby(train_ratings["rater_id"]).mean().reindex(eligible_users)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].bar(dist.index.astype(str), 100 * dist.values, color="steelblue")
axes[0].set_xlabel("rating")
axes[0].set_ylabel("% of ratings")
axes[0].set_title("Rating distribution")
axes[1].hist(user_ge4, bins=40, color="steelblue", edgecolor="white")
axes[1].set_xlabel("share of the user's ratings that are >= 4")
axes[1].set_ylabel("users")
axes[1].set_title("Per-user share of 4-5 star ratings (eligible users)")
fig.tight_layout()
plt.show()

user_ge4.quantile([0.05, 0.25, 0.5, 0.75, 0.95]).round(2).rename("share of ratings >= 4")

**Observations**

- **Ratings lean positive:** 4 stars is the most common rating (35%), then 3 (29%) and 5 (22%); only 14% are 1–2 stars. 57% of all ratings are 4 or 5. Users mostly rate what they chose to watch, so the ratings describe films users selected, not random films.
- **Users differ a lot in how often they give 4–5 stars:** the median user gives 4–5 stars to 58% of their ratings, but a quarter of users are below 46% and a quarter above 71%; 5% of users are below 28% and 5% above 88%.
- **Consequence for the threshold:** a single fixed cut-off on the predicted rating (e.g. "recommend if ≥ 4") would recommend most movies to generous users and few to harsh ones. Since the model uses the user's own statistics, its predicted rating is already on each user's scale; the threshold choice should be checked separately for harsh and generous users, or expressed per user (e.g. recommend the user's top-N predicted movies).

## 8. Train-only aggregates

Saved for clustering (`08`) and the model (`09`). Everything is computed from ratings of **training movies** only.

**Movie statistics** (training movies): `avg_rating`, `rating_std` (polarization, RQ2) and `n_ratings`.

In [ ]:
movie_stats = (
    train_ratings.groupby("netflix_movie_id")["rating"]
    .agg(avg_rating="mean", rating_std="std", n_ratings="size")
    .reset_index()
)
movie_stats.to_csv(PROCESSED / "movie_stats_train.csv", index=False)
movie_stats.describe().round(3)

**Director and cast track records** (same definition as `05_eda_movies_raw` section 9): the average `avg_rating` of a person's *other* films, shrunk toward the global mean with m = 3, so one lucky film cannot produce an extreme score:

$$\text{track record} = \frac{\text{sum of avg ratings of the other films} + m \cdot \text{global mean}}{\text{number of other films} + m}$$

Only **training** films count as "other films":
- training movie: the person's other training films (leave-one-out, the movie's own rating is excluded);
- validation / test movie: all of the person's training films (its own rating is never known).

`cast_track_record` averages the actors' scores over the first 5 billed actors.

In [ ]:
SHRINK = 3

train_avg = movie_stats.set_index("netflix_movie_id")["avg_rating"]
TRACK_GLOBAL_MEAN = train_avg.mean()


def track_record(pairs):
    """pairs: one row per (netflix_movie_id, person). Returns the shrunk mean of the person's other training films."""
    own = pairs["netflix_movie_id"].map(train_avg)  # NaN for validation / test movies
    totals = pd.DataFrame({"person": pairs["person"], "own": own}).dropna().groupby("person")["own"].agg(["sum", "count"])
    j = pairs.join(totals, on="person").fillna({"sum": 0, "count": 0})
    other_sum = j["sum"] - own.fillna(0)
    other_n = j["count"] - own.notna().astype(int)
    return (other_sum + SHRINK * TRACK_GLOBAL_MEAN) / (other_n + SHRINK)


director_pairs = movies[["netflix_movie_id", "director"]].rename(columns={"director": "person"})
director_tr = track_record(director_pairs).set_axis(movies["netflix_movie_id"])

cast_pairs = movies[["netflix_movie_id", "cast_top5"]].explode("cast_top5").rename(columns={"cast_top5": "person"})
cast_pairs = cast_pairs.dropna(subset=["person"]).reset_index(drop=True)
cast_tr = track_record(cast_pairs).groupby(cast_pairs["netflix_movie_id"]).mean()

track_records = pd.DataFrame({
    "netflix_movie_id": movies["netflix_movie_id"],
    "director_track_record": director_tr.to_numpy(),
    "cast_track_record": movies["netflix_movie_id"].map(cast_tr).fillna(TRACK_GLOBAL_MEAN).to_numpy(),
})
track_records.to_csv(PROCESSED / "track_records.csv", index=False)

Sanity check: Spearman correlation of the track records with the movie's average rating. On validation / test movies it shows the honest, cold-start strength of the feature (the EDA found ~0.2 for directors and ~0.3 for the cast on all movies).

In [ ]:
check = track_records.merge(movies[["netflix_movie_id", "split", "avg_rating"]], on="netflix_movie_id")
pd.DataFrame({
    split: {
        c: part[c].corr(part["avg_rating"], method="spearman") for c in ["director_track_record", "cast_track_record"]
    }
    for split, part in check.groupby("split")
}).T.loc[["train", "val", "test"]].round(3)

**User statistics** (eligible users, ratings of training movies only), used directly as model features:

| column | meaning |
|---|---|
| `user_mean` | the user's average rating: how generous they are |
| `user_std` | standard deviation of the user's ratings: does the user use the whole 1–5 scale or give nearly the same rating to everything |
| `user_n` | number of training movies the user rated: how active they are, and how reliable their other statistics are |
| `user_pct_ge4` | share of the user's ratings that are **g**reater than or **e**qual to **4** stars (0.7 = 70% of their ratings are 4 or 5): how often the user "likes" a film, the same event the "add to list" decision targets |

`user_pct_ge4` is closely related to `user_mean` (correlation ≈ 0.95), but not the same: a user rating 3, 3, 3, 5 and a user rating 3, 4, 4, 3 both have mean 3.5, but the first "likes" 25% of their films and the second 50%. If the model shows it adds nothing beyond `user_mean`, it can be dropped in `09`.

In [ ]:
eligible_train = train_ratings[train_ratings["rater_id"].isin(eligible_users)]
user_stats = eligible_train.groupby("rater_id")["rating"].agg(user_mean="mean", user_std="std", user_n="size")
user_stats["user_pct_ge4"] = (eligible_train["rating"] >= 4).groupby(eligible_train["rater_id"]).mean()
user_stats = user_stats.reset_index()
user_stats.to_csv(PROCESSED / "user_stats_train.csv", index=False)
user_stats.describe().round(3)

## 9. Leakage checks and summary of outputs

In [ ]:
train_set = set(train_ids)
assert set(movie_stats["netflix_movie_id"]) <= train_set
assert user_stats["user_n"].min() >= MIN_TRAIN_RATINGS
assert len(track_records) == len(movies) and track_records.notna().all().all()

for name in ["movie_split.csv", "movie_stats_train.csv", "track_records.csv", "user_stats_train.csv"]:
    df = pd.read_csv(PROCESSED / name, nrows=0)
    print(f"{name}: columns {list(df.columns)}")

**Observations**

- **Split:** 919 / 197 / 198 movies (69% / 16% / 15% of ratings). The three parts are balanced on average rating (3.44–3.46), popularity, year, runtime, `log_roi` and US share; the documentaries fell 7 / 2 / 1.
- **Users:** with `MIN_TRAIN_RATINGS = 20`, 332,632 users (72%) are kept and they hold 96% of the validation / test ratings. Median: 69 training ratings per eligible user.
- **User vs. movie:** the user's mean explains more of a rating (R² 0.16) than the movie's mean (0.10); together 0.26. Generosity differs a lot between users, so user statistics are essential model features, and they are kept *out* of the clustering so personas reflect taste, not generosity.
- **Genre taste:** for every genre the real spread across users exceeds the shuffled (noise-only) spread. Strongest for **action, documentary, comedy, horror, drama and science fiction** (1.2–1.4×); weakest for mystery, war, crime and history (~1.05–1.08×): war and history films are rated above a user's own mean by almost everyone, and mystery and crime are close to neutral for everyone, so these genres do not divide users. This supports persona clustering and suggests which genres will separate the personas.
- **Ratings:** 57% of ratings are 4 or 5 stars and 22% are 5 stars, and the per-user share of 4–5 stars varies widely, so a single "add to list" threshold on the predicted rating will mean different things for generous and harsh users.
- **Track records:** Spearman with the movie's average rating is 0.23 / 0.34 (director / cast) on training movies (leave-one-out) and 0.20 / 0.29 on test movies. On validation movies it is 0.14 / 0.14: with ~200 movies per part the estimate is noisy (standard error ≈ 0.07), so the cold-start strength of these features is modest but real.